In [3]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar

from chggen.common.data_utils import get_scaler
from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_data.datamodule import CrystDataModule
from chggen.pl_modules.model_egnn import CHGGen

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Dataset.
train_dataset = CHGNetDataset(
    path= '/home/xzdai/ceder_group/material_dircovery/chggen_old/data/perov_5/test_zpc.csv', # train.csv
    name = 'train_perov',
    prop_list = ['heat_all'],
)

val_dataset = CHGNetDataset(
    path= '/home/xzdai/ceder_group/material_dircovery/chggen_old/data/perov_5/test_zpc.csv', # val.csv
    name = 'val_perov',
    prop_list = ['heat_all'],
)

# Compute lattice scaler.
lattice_scaler = get_scaler(dataset=train_dataset)

100%|██████████| 50/50 [00:00<00:00, 243.69it/s]
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:654: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:619: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [5]:
datamodule = CrystDataModule(
    train_dataset = train_dataset,
    val_dataset = val_dataset,
    num_workers = 8,
    batch_size = 16,
)

In [11]:
batch = next(iter(datamodule.train_dataloader()))

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [16]:
batch.frac_coords.shape

torch.Size([80, 3])

In [5]:
model_hparams = {'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1, # predict the multiple property 
                'load_pretrain': True, 'fc_num_layers': 1, 
                'sigma_F_begin': 10.0, 'sigma_F_end': 0.01, 
                'sigma_L_begin': 1.0, 'sigma_L_end': 0.01, 
                'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 20, # should be larger than the training set.
                'num_noise_level': 1, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'egnn'}

chggen = CHGGen(
    lattice_scaler = lattice_scaler, hparams_dict = model_hparams
)

CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


In [6]:
# Define the checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath= './test_models/perov/',
    filename='{epoch}',        # Save the checkpoint after every epoch
    save_top_k=-1,            # Set to -1 to save all checkpoints
    save_last=True,           # Save the last model too, useful for resuming
    every_n_train_steps=100,     # Save every epoch (assuming you're validating every epoch)
    verbose=True              # Print save messages for debugging
)

ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.


In [7]:
trainer = pl.Trainer(
    accelerator = "gpu", 
    devices = [0],
    max_epochs = 10,
    callbacks = [checkpoint_callback, TQDMProgressBar(refresh_rate = 1)],
    #  strategy = 'ddp_find_unused_parameters_true',  # multi-GPU training                    
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: UserWarning: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
  warning_cache.warn(


In [8]:
trainer.fit(model = chggen, datamodule = datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5]

  | Name           | Type           | Params
--------------------------------------------------
0 | encoder        | CHGNet_encoder | 400 K 
1 | decoder        | EGNNDecoder    | 33.5 K
2 | fc_mu          | Linear         | 4.2 K 
3 | fc_var         | Linear         | 4.2 K 
4 | fc_latent_proj | Sequential     | 16.8 K
5 | fc_num_atoms   | Sequential     | 11.0 K
6 | fc_lattice     | Sequential     | 9.1 K 
7 | fc_composition | Sequential     | 20.4 K
8 | fc_property    | Sequential     | 8.4 K 
--------------------------------------------------
107 K     Trainable params
400 K     Non-trainable params
508 K     Total params
2.032     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:630: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [ ]:
list(vars(batch)['_store'].keys())

['edge_index',
 'crys_graph',
 'frac_coords',
 'atom_types',
 'lengths',
 'angles',
 'lattices',
 'num_atoms',
 'properties',
 'batch',
 'ptr']

In [ ]:
graphs = [g for g in batch.crys_graph]

In [ ]:
mu, logvar, z = chggen.encode(graphs)

In [ ]:
chggen.sigmas_F.shape[0]

1